In [3]:
# Import the packages we need

import pandas as pd
from sklearn.model_selection import train_test_split

# Load the dataset, saved in current folder as data.csv
file_path = "data.csv"
df = pd.read_csv(file_path)

# Check the loaded data
print("Dataset shape:", df.shape)
print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))
display(df.head())

# Separate the 86 permission features from the target label
X = df.drop(columns="Result")
y = df["Result"]

print("Number of features:", X.shape[1])
print("Number of observations:", X.shape[0])
print("\nClass counts:")
print(y.value_counts()) # number of 0s and number of 1s

# Create a group label for each unique permission pattern
# permission group holds each data point with their permissions joined together with "_"
# ex: 0_1_0_1_1
permission_groups = X.astype(str).agg("_".join, axis=1)

print(
    "\nNumber of unique permission patterns:",
    permission_groups.nunique()
)

# Count apps that share the same permission pattern
group_counts = permission_groups.value_counts()

print("\nMost common permission patterns:")
print(group_counts.head(10))

# Count rows identical across all features and the target
# Sum gives you number of rows which are a duplicate of an earlier row before it
print("\nNumber of duplicate rows:", df.duplicated().sum())

# Count extra apps beyond the first for each permission pattern
# Gives count of how many duplicate permission patterns there are
print(
    "Number of repeated permission-pattern rows:",
    len(permission_groups) - permission_groups.nunique()
)

# Check whether any permission pattern has both labels
# groupby organizes number of 0s and 1s for each unique permission set
# nunique will return 2 if it has both kinds of labels
# Isolates those apps into its own group and prints the total count of them
label_counts = y.groupby(permission_groups).nunique()
conflicting_groups = label_counts[label_counts > 1]
# display(conflicting_groups.head())

print(
    "\nNumber of permission patterns with BOTH labels:",
    len(conflicting_groups)
)

# Create one row per permission pattern for group-based splitting.
# group_labels contains all the unique permission patterns
# Mark patterns that have both labels as -1 (denotes conflicting labels), 1 otherwise
group_labels = y.groupby(permission_groups, sort=False).agg(
    lambda labels: labels.iloc[0] if labels.nunique() == 1 else -1
)

# deduplicate permission patterns
group_data = X.groupby(permission_groups, sort=False).first()
group_data["Result"] = group_labels 

print("\nGroup-level label counts:")
print(group_data["Result"].value_counts())
print("Group-level dataset shape:", group_data.shape)

# print(group_labels.index.shape)
# print("First 10 permission patterns:")
# print(group_labels.index[3])
# Split unique permission patterns into training (70%) and testing (30%)
train_groups, test_groups = train_test_split(
    group_labels.index.tolist(),
    test_size=0.30,
    random_state=42
)

# Put each original app in the split according to its permission pattern
# test and train don't share any permission patterns, so that the model can't learn an exact pattern
#  and then be able to apply to test
train_mask = permission_groups.isin(train_groups)
test_mask = permission_groups.isin(test_groups)

train_data = df.loc[train_mask].copy()
test_data = df.loc[test_mask].copy()

print("Training rows:", len(train_data))
print("Test rows:", len(test_data))
print(
    "Permission patterns shared between splits:",
    len(set(train_groups) & set(test_groups))
)

Dataset shape: (29332, 87)
Number of rows: 29332
Number of columns: 87


,android.permission.GET_ACCOUNTS,com.sonyericsson.home.permission.BROADCAST_BADGE,android.permission.READ_PROFILE,android.permission.MANAGE_ACCOUNTS,android.permission.WRITE_SYNC_SETTINGS,android.permission.READ_EXTERNAL_STORAGE,android.permission.RECEIVE_SMS,com.android.launcher.permission.READ_SETTINGS,android.permission.WRITE_SETTINGS,com.google.android.providers.gsf.permission.READ_GSERVICES,...,com.android.launcher.permission.UNINSTALL_SHORTCUT,com.sec.android.iap.permission.BILLING,com.htc.launcher.permission.UPDATE_SHORTCUT,com.sec.android.provider.badge.permission.WRITE,android.permission.ACCESS_NETWORK_STATE,com.google.android.finsky.permission.BIND_GET_INSTALL_REFERRER_SERVICE,com.huawei.android.launcher.permission.READ_SETTINGS,android.permission.READ_SMS,android.permission.PROCESS_INCOMING_CALLS,Result
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,1,0,0,0,0


Number of features: 86
Number of observations: 29332

Class counts:
Result
1    14700
0    14632
Name: count, dtype: int64

Number of unique permission patterns: 7395

Most common permission patterns:
0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_1_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_1_0_0_0_0    2009
0_0_1_0_0_0_0_0_1_0_0_1_1_0_0_0_0_0_0_0_1_1_0_0_0_1_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_1_0_0_0_1_0_0_0_0_1_0_0_1_1_1_0_1_0_0_0_0_0_0_0_0_0_0_0_1_0_0_0_0    1171
0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_1_0_0_0_1_0_0_0_1_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_1_0_0_0_0_0_0_0_0_0_0_0_1_0_1_0_0_0_0_0_1_0_0_0_0_0_0_0_1_0_0_0_0     555
0_0_0_0_0_0_1_0_0_0_0_1_1_0_0_0_0_0_0_1_0_1_0_0_0_1_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_1_0_0_0_0_0_0_0_0_0_1_0_0_0_0_0_0_0_1_0_0_1_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_1_0_0_1_0     546
0_0_0_0_0_0_0_0_1_0_0_0_1_0_0_0_0_0_0_0_0_1_0_0_0_1_0_0_0_0_1_0_0_0_1_0_0_0_0_0